<a href="https://colab.research.google.com/github/acerNZ/HAL/blob/master/ReqTo_BDD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ========================================================
#  FULL MoSCoW DOC → Gherkin BDD (Smart Extractor)
#  Handles: Full documents, numbered items, (W), custom labels
#  Google Colab Ready
# ========================================================

import re
import pandas as pd
import io
import base64
import ipywidgets as widgets
from IPython.display import display, HTML

# ------------------- EXTRACT NUMBERED REQUIREMENTS -------------------
def extract_requirements(text):
    """Extract each numbered requirement like: 1. (M) Description"""
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    requirements = []
    current = {"priority": "M", "text": "", "raw": ""}

    for line in lines:
        # Match: 1. (M) Some text
        match = re.match(r"^(\d+)\.\s*\*\*?\(([MSCW])\)\*\*?\s*(.+)", line)
        if match:
            # Save previous
            if current["text"]:
                requirements.append(current.copy())
            # Start new
            num, prio, desc = match.groups()
            current = {"priority": prio, "text": desc.strip(), "raw": line, "custom": {}}
        else:
            # Append to current requirement
            if current["text"] and ":" in line:
                key, val = line.split(":", 1)
                current["custom"][key.strip().lower()] = val.strip()
            elif current["text"]:
                current["text"] += " " + line.strip()

    if current["text"]:
        requirements.append(current)

    return requirements

# ------------------- SMART GHERKIN GENERATOR -------------------
def generate_scenarios(req):
    scenarios = []
    prio = req["priority"]
    text = req["text"]
    c = req["custom"]

    # Default role/context
    role = c.get("role", "an internal user")
    context = c.get("context", "the relevant page")
    action = c.get("action", "perform the action")
    expected = c.get("expected", None)

    # === 1. Happy Path (only if not (W)) ===
    if prio != "W" and any(word in text.lower() for word in ["display", "allow", "save", "show", "retrieve"]):
        verb = "display" if "display" in text.lower() else \
               "allow" if "allow" in text.lower() else \
               "save" if "save" in text.lower() else "see"
        given = f"Given I am {role}"
        if "context" in c: given += f"\nAnd I am on {context}"
        when = f"When I {action}" if "action" in c else f"When I view the page"
        then = f"Then the system should {verb} {text.lower()}"
        scenarios.append(f"{given}\n{when}\n{then}")

    # === 2. (W) Out of Scope → Negative Scenario ===
    if prio == "W":
        action_word = text.lower().split()[0]  # e.g., "editing" → "edit"
        scenarios.append(
            f"Given I am {role}\n"
            f"When I attempt to {action_word} locked fields\n"
            f"Then the feature should not be available\n"
            f"And the system should prevent any changes"
        )
        # Specific for your case
        if "editing locked fields" in text.lower():
            scenarios.append(
                f"Given I am {role}\n"
                f"And a record has been saved\n"
                f"When I try to edit Place, Role, or Start Date\n"
                f"Then these fields should be locked and uneditable"
            )
        if "overlap validation" in text.lower():
            scenarios.append(
                f"Given I am {role}\n"
                f"And two active place assignments exist with overlapping dates\n"
                f"Then the system should allow it (no validation enforced)"
            )
        if "reassigning" in text.lower():
            scenarios.append(
                f"Given I am {role}\n"
                f"When I try to replace an active place assignment\n"
                f"Then the system should not support replacement\n"
                f"And only allow end-dating"
            )

    # === 3. Custom Labels ===
    if "validation" in c:
        scenarios.append(f"# Validation: {c['validation']}")
    if "error" in c:
        scenarios.append(f"# Error Case: {c['error']}")

    return "\n\n".join(scenarios) if scenarios else "No scenario generated."

# ------------------- MAIN PROCESSING -------------------
uploader = widgets.FileUpload(accept='.xlsx,.csv', multiple=False, description="Upload Excel/CSV")
output = widgets.Output()

def process_file(change):
    with output:
        output.clear_output()
        if not uploader.value:
            print("No file uploaded.")
            return

        info = list(uploader.value.values())[0]
        name = info['metadata']['name']
        content = info['content']
        file_io = io.BytesIO(content)

        try:
            df = pd.read_excel(file_io) if name.endswith('.xlsx') else pd.read_csv(file_io)
        except Exception as e:
            print(f"Error reading file: {e}")
            return

        if 'Requirements' not in df.columns:
            print(f"Column 'Requirements' not found! Found: {list(df.columns)}")
            return

        gherkin_list = []
        for cell in df['Requirements']:
            if pd.isna(cell) or not str(cell).strip():
                gherkin_list.append("")
                continue

            reqs = extract_requirements(str(cell))
            all_gherkin = []
            for req in reqs:
                gherkin = generate_scenarios(req)
                all_gherkin.append(f"# {req['raw']}\n{gherkin}")
            gherkin_list.append("\n\n".join(all_gherkin))

        df['Acceptance Criteria'] = gherkin_list

        print("SUCCESS! Preview:")
        display(df[['Requirements', 'Acceptance Criteria']].head(2))

        # Download
        buf = io.BytesIO()
        out_name = f"Gherkin_{name}"
        if name.endswith('.xlsx'):
            df.to_excel(buf, index=False, engine='openpyxl')
            ext = 'xlsx'
        else:
            df.to_csv(buf, index=False)
            ext = 'csv'
        buf.seek(0)

        link = f'<a href="data:{"application/vnd.openxmlformats-officedocument.spreadsheetml.sheet" if ext=="xlsx" else "text/csv"};base64,{base64.b64encode(buf.read()).decode()}" download="{out_name}">Download Gherkin Output</a>'
        display(HTML(f"<br><strong>Download:</strong> {link}"))

uploader.observe(process_file, names='value')
display(uploader, output)